# 12. CU-BEMS: planted-anomaly test (task T32)
Train 1 Jul 2018 - 30 Jun 2019, validation Jul-Sep 2019, test Oct-Dec 2019. Models A', B', C' use the midterm settings (calendar + series id; B' adds lag 24 and 168; C' adds lag 1). Problems are planted **only in the test period** (nothing is tuned on planted data), 16 per series (lengths 1, 3, 8, 24 h x shapes x1.3, x1.6, x2.0, drop to 5%), 20 random placements. Scaling and cut-offs come from the clean validation period.

Planted problems are our own simple design: results must not be quoted as real-world performance.

In [ ]:
"""CU-BEMS planted-anomaly test (T31/T32). Train 2018-07-01..2019-06-30, val 2019-07..09, test 2019-10..12.
Anomalies are planted ONLY in the test period (nothing is tuned on planted data)."""
import numpy as np, pandas as pd, json, sys, os
os.makedirs('../results/cu_bems',exist_ok=True)
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
W=pd.read_csv('../data/cu_bems_hourly/cu_bems_hourly_wide.csv.gz',index_col=0,parse_dates=True)
SER=list(W.columns); idx=W.index
TR=(idx<'2019-07-01'); VA=(idx>='2019-07-01')&(idx<'2019-10-01'); TE=(idx>='2019-10-01')
hour=idx.hour.values; dow=idx.dayofweek.values; wkd=(dow>=5).astype(int)
def feats(X, lags=(), sid=None):
    """X: array (T,) kwh for one series -> feature frame"""
    s=pd.Series(X,index=idx)
    d=pd.DataFrame({'hour':hour,'dow':dow,'wkd':wkd,'sid':sid},index=idx)
    for L in lags: d[f'lag_{L}']=s.shift(L).values
    return d
SETS={'A':(),'B':(24,168),'C':(1,24,168)}
def build(Wm,lags):
    return pd.concat([feats(Wm[s].values,lags,i) for i,s in enumerate(SER)],keys=SER)
def targ(Wm): return np.concatenate([np.log1p(Wm[s].values) for s in SER])
Y0=targ(W)
mask_tr=np.concatenate([TR]*len(SER))&~np.isnan(Y0)
models={}
for k,l in SETS.items():
    X=build(W,l).values.astype(float)
    m=HistGradientBoostingRegressor(max_iter=600,learning_rate=0.05,max_leaf_nodes=127,min_samples_leaf=100,l2_regularization=1.0,random_state=0,categorical_features=[3])
    m.fit(X[mask_tr],Y0[mask_tr]); models[k]=m; print('fit',k,mask_tr.sum(),flush=True)
# baselines
def typical(Wm):
    out={}
    for s in SER:
        tr=pd.DataFrame({'v':W[s][TR].values,'h':hour[TR],'w':wkd[TR]}).dropna()
        med=tr.groupby(['h','w']).v.median(); fb=tr.groupby('h').v.median()
        out[s]=np.array([med.get((h,w),fb.get(h,np.nan)) for h,w in zip(hour,wkd)])
    return out
TYP=typical(W)
def predictions(Wm):
    P={}
    for k,l in SETS.items():
        X=build(Wm,l).values.astype(float); P[k]=models[k].predict(X).reshape(len(SER),-1)
    P['last']=np.log1p(np.vstack([np.roll(Wm[s].values,1) for s in SER])); P['last'][:,0]=np.nan
    P['typ']=np.log1p(np.vstack([TYP[s] for s in SER]))
    return P
def errors(Wm,P):
    y=np.log1p(np.vstack([Wm[s].values for s in SER]))
    return {k:y-P[k] for k in P}
# clean validation scaling
P0=predictions(W); E0=errors(W,P0)
scale={}
for k in E0:
    e=E0[k]; med=np.nanmedian(np.where(VA,e,np.nan),axis=1); mad=np.nanmedian(np.abs(np.where(VA,e,np.nan)-med[:,None]),axis=1)*1.4826
    scale[k]=(med,np.maximum(mad,0.05))
def scores(E,k): m,s=scale[k]; return np.abs(E[k]-m[:,None])/s[:,None]
S0={k:scores(E0,k) for k in E0}
cut={k:np.nanquantile(S0[k][:,VA],0.995) for k in S0}   # pooled midterm-style rule: 0.5% of clean val
# isolation forest (variant 2 features) one per series on training rows
def iffeat(Wm,i):
    s=pd.Series(Wm[SER[i]].values,index=idx); l=np.log1p(s)
    return pd.DataFrame({'lv':l,'d1':l-l.shift(1),'d24':l-l.shift(24),'h':hour,'w':wkd},index=idx)
ifm=[]
for i in range(len(SER)):
    F=iffeat(W,i); ok=F.notna().all(axis=1).values&TR
    ifm.append(IsolationForest(n_estimators=100,max_samples=256,random_state=0).fit(F[ok]))
def ifscores(Wm):
    out=np.full((len(SER),len(idx)),np.nan)
    for i in range(len(SER)):
        F=iffeat(Wm,i); ok=F.notna().all(axis=1).values
        out[i,ok]=-ifm[i].score_samples(F[ok])
    return out
IF0=ifscores(W); ifcut=np.array([np.nanquantile(IF0[i,VA],0.995) for i in range(len(SER))])
# planting
LENS=[1,3,8,24]; REPS={1:4,3:4,8:4,24:4}; SHAPES=[('x1.3',1.3),('x1.6',1.6),('x2.0',2.0),('drop',0.05)]
def plant(seed):
    rng=np.random.default_rng(seed); Wm=W.copy(); ev=[]
    tidx=np.where(TE)[0]
    for s in SER:
        v=Wm[s].values.copy(); ok=~np.isnan(v); taken=np.zeros(len(idx),bool); n=0
        for L in sorted(LENS,reverse=True):
            for r in range(REPS[L]):
                for _ in range(2000):
                    st=rng.choice(tidx)
                    if st+L>len(idx) or not TE[st+L-1]: continue
                    lo=max(0,st-48); hi=min(len(idx),st+L+48)
                    if taken[lo:hi].any() or not ok[st:st+L].all(): continue
                    nm,f=SHAPES[r]; n+=1
                    v[st:st+L]=v[st:st+L]*f; taken[st:st+L]=True
                    ev.append((s,st,L,nm,f)); break
        Wm[s]=v
    return Wm,pd.DataFrame(ev,columns=['series','start','length','shape','factor'])
METH=['last','typ','A','B','C','IF']
NAMES={'last':'Same as last hour','typ':'Historical typical value','A':"Model A'",'B':"Model B'",'C':"Model C'",'IF':'Isolation Forest (raw)'}
def method_scores(Wm):
    P=predictions(Wm); E=errors(Wm,P); S={k:scores(E,k) for k in E}; S['IF']=ifscores(Wm); return S
def evaluate(S,ev,rates=(0.005,0.01,0.02,0.05)):
    rows=[];pos=[]
    sid={s:i for i,s in enumerate(SER)}
    for k in METH:
        sc=S[k]
        # per-series equal alarm rate
        for q in rates:
            flag=np.zeros_like(sc,bool)
            for i in range(len(SER)):
                x=np.where(TE,sc[i],np.nan); n=np.isfinite(x).sum()
                if n==0: continue
                th=np.nanquantile(x,1-q); flag[i]=np.nan_to_num(x,nan=-np.inf)>=th
            rows+=score_flags(k,'equal_rate_%g'%(q*100),flag,ev,sid,sc)
        # frozen midterm-style cut-off (0.5% of clean val)
        flag=np.zeros_like(sc,bool)
        for i in range(len(SER)):
            th=cut[k] if k!='IF' else ifcut[i]
            flag[i]=np.nan_to_num(np.where(TE,sc[i],np.nan),nan=-np.inf)>=th
        rows+=score_flags(k,'frozen_cut',flag,ev,sid,sc)
        if k in ('last','C','B','IF'):
            pass
    return rows
def score_flags(k,rule,flag,ev,sid,sc):
    out=[]; nvalid=np.isfinite(np.where(TE,sc,np.nan)).sum()
    rate=flag[:,TE].sum()/nvalid
    for _,e in ev.iterrows():
        i=sid[e['series']]; seg=flag[i,e.start:e.start+e.length]
        first=int(np.argmax(seg)) if seg.any() else -1
        out.append(dict(method=k,rule=rule,length=e.length,shape=e['shape'],found=int(seg.any()),hour_share=seg.mean(),delay=first,flag_rate=rate))
    return out
NS=20

allr=[];evs=[];pp=[]
for seed in range(NS):
    Wm,ev=plant(1000+seed); S=method_scores(Wm)
    r=pd.DataFrame(evaluate(S,ev)); r['seed']=seed; allr.append(r); ev['seed']=seed; evs.append(ev)
    # persistence for 24-h events at 1%
    sid={s:i for i,s in enumerate(SER)}
    for k in METH:
        sc=S[k]; flag=np.zeros_like(sc,bool)
        for i in range(len(SER)):
            x=np.where(TE,sc[i],np.nan); th=np.nanquantile(x,0.99); flag[i]=np.nan_to_num(x,nan=-np.inf)>=th
        for _,e in ev[ev.length==24].iterrows():
            seg=flag[sid[e.series],e.start:e.start+24]
            for h in range(24): pp.append((seed,k,h,int(seg[h])))
    print('seed',seed,len(ev),flush=True)
R=pd.concat(allr); R.to_csv('../results/cu_bems/cubems_event_results.csv.gz',index=False)
pd.concat(evs).to_csv('../results/cu_bems/cubems_planted_events.csv',index=False)
pd.DataFrame(pp,columns=['seed','method','hour_since_start','flagged']).to_csv('../results/cu_bems/cubems_persistence_24h.csv',index=False)
json.dump({'series':SER,'cut_pooled_0.5pct':{k:float(v) for k,v in cut.items()},'if_cut':ifcut.tolist(),'scale_floor':0.05},open('../results/cu_bems/cubems_cutoffs.json','w'),indent=1)


In [ ]:
import pandas as pd, numpy as np
import matplotlib; import matplotlib.pyplot as plt
OUT='../results/cu_bems/'
R=pd.read_csv(OUT+'cubems_event_results.csv.gz').rename(columns={'shape':'kind'})
N={'last':'Same as last hour','typ':'Historical typical value','A':"Model A'",'B':"Model B'",'C':"Model C'",'IF':'Isolation Forest'}
rows=[]
for rule in R.rule.unique():
    for by in ['length','kind']:
        g=R[R.rule==rule].groupby(['seed','method',by]).found.mean().reset_index()
        m=g.groupby(['method',by]).found.agg(['mean','std']).reset_index(); m.columns=['method','group_value','recall','sd']; m['group']=by; m['rule']=rule; rows.append(m)
    g=R[R.rule==rule].groupby(['seed','method']).found.mean().reset_index()
    m=g.groupby('method').found.agg(['mean','std']).reset_index(); m.columns=['method','recall','sd']; m['group']='all'; m['group_value']='all'; m['rule']=rule; rows.append(m)
T=pd.concat(rows); T['method']=T.method.map(N); T.to_csv(OUT+'cubems_recall_tables.csv',index=False)
P=pd.read_csv(OUT+'cubems_persistence_24h.csv')
p=P.groupby(['method','hour_since_start']).flagged.mean().unstack(0)*100; p.to_csv(OUT+'cubems_persistence_24h_rate.csv')
fig,ax=plt.subplots(figsize=(6.5,3.8))
for k,c in [('last','#c0504d'),('C','#1f6f8b'),('IF','#6a994e')]:
    ax.plot(p.index,p[k].rolling(3,center=True,min_periods=1).mean(),label=N[k],color=c,lw=2)
ax.axhline(1,color='#999',ls='--',lw=1); ax.text(23,1.3,'overall alarm rate (1%)',ha='right',fontsize=8,color='#666')
ax.set_xlabel('Hours since the start of a planted 24-hour anomaly'); ax.set_ylabel('% of events flagged at that hour')
ax.set_title('CU-BEMS: flagging during a 24-hour anomaly (1% of hours flagged)',fontsize=9); ax.legend(frameon=False); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig(OUT+'cubems_persistence_24h.png',dpi=160)
print(T[(T.group=='all')].pivot(index='method',columns='rule',values='recall').round(3))